# Black Grouse 2030 Land-Use Scenario Project

## Stage 4: Core-Habitat Landscape Metrics

This notebook quantifies how the spatial configuration of core habitat changed between 1990, 2012, and 2024.

The same metrics will later be applied to the alternative 2030 restoration scenarios, allowing scenarios with equal restored habitat area to be compared according to habitat fragmentation and configuration.

Metrics included:

- Total core-habitat area
- Number of habitat patches
- Patch density
- Mean patch area
- Median patch area
- Largest patch area
- Largest Patch Index
- Total habitat-edge length
- Edge density

Connectivity metrics will be calculated separately in Stage 7.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

from scipy import ndimage

print("Packages imported successfully.")

Packages imported successfully.


In [2]:
PROJECT_DIR = Path.cwd()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

HARMONISED_RASTERS = {
    1990: PROCESSED_DIR / "HGN1990_harmonised_25m.tif",
    2012: PROCESSED_DIR / "LGN7_2012_harmonised_25m.tif",
    2024: PROCESSED_DIR / "LGN2024_harmonised_25m.tif",
}

CORE_HABITAT_CLASS = 2

for year, raster_path in HARMONISED_RASTERS.items():

    status = "FOUND" if raster_path.exists() else "MISSING"

    print(f"{year}: {status}")
    print(raster_path)

1990: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\HGN1990_harmonised_25m.tif
2012: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\LGN7_2012_harmonised_25m.tif
2024: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\LGN2024_harmonised_25m.tif


In [3]:
landuse_arrays = {}
raster_profiles = {}

reference_shape = None
reference_transform = None
reference_crs = None
reference_mask = None

for year, raster_path in HARMONISED_RASTERS.items():

    with rasterio.open(raster_path) as src:

        raster_data = src.read(1)

        landuse_arrays[year] = raster_data
        raster_profiles[year] = src.profile.copy()

        current_mask = raster_data != 0

        if reference_shape is None:

            reference_shape = raster_data.shape
            reference_transform = src.transform
            reference_crs = src.crs
            reference_mask = current_mask.copy()

        else:

            if raster_data.shape != reference_shape:
                raise ValueError(
                    f"The {year} raster has a different shape."
                )

            if src.transform != reference_transform:
                raise ValueError(
                    f"The {year} raster has a different transform."
                )

            if src.crs != reference_crs:
                raise ValueError(
                    f"The {year} raster has a different CRS."
                )

            if not np.array_equal(
                current_mask,
                reference_mask,
            ):
                raise ValueError(
                    f"The {year} raster has a different analysis mask."
                )

        observed_classes = sorted(
            np.unique(
                raster_data[current_mask]
            ).astype(int).tolist()
        )

        core_pixels = int(
            np.sum(raster_data == CORE_HABITAT_CLASS)
        )

        print(f"\nYear: {year}")
        print(f"Shape: {raster_data.shape}")
        print(f"CRS: {src.crs}")
        print(f"Resolution: {src.res}")
        print(f"Observed classes: {observed_classes}")
        print(f"Core-habitat pixels: {core_pixels}")


PIXEL_SIZE_METRES = abs(
    reference_transform.a
)

PIXEL_AREA_M2 = (
    PIXEL_SIZE_METRES
    * PIXEL_SIZE_METRES
)

PIXEL_AREA_KM2 = (
    PIXEL_AREA_M2
    / 1_000_000
)

ANALYSIS_AREA_KM2 = (
    reference_mask.sum()
    * PIXEL_AREA_KM2
)

print("\nAll rasters use the same analysis grid.")
print(f"Pixel size: {PIXEL_SIZE_METRES:.0f} m")
print(f"Analysis area: {ANALYSIS_AREA_KM2:.2f} km²")


Year: 1990
Shape: (1592, 1023)
CRS: EPSG:28992
Resolution: (25.0, 25.0)
Observed classes: [1, 2, 3, 4, 5]
Core-habitat pixels: 34420

Year: 2012
Shape: (1592, 1023)
CRS: EPSG:28992
Resolution: (25.0, 25.0)
Observed classes: [1, 2, 3, 4, 5]
Core-habitat pixels: 68403

Year: 2024
Shape: (1592, 1023)
CRS: EPSG:28992
Resolution: (25.0, 25.0)
Observed classes: [1, 2, 3, 4, 5]
Core-habitat pixels: 87052

All rasters use the same analysis grid.
Pixel size: 25 m
Analysis area: 762.88 km²


In [4]:
def calculate_core_habitat_metrics(
    raster_data,
    valid_mask,
    year,
):
    """
    Calculate core-habitat configuration metrics.

    Habitat patches are identified using eight-neighbour
    connectivity, meaning that diagonally touching habitat
    pixels belong to the same patch.

    Edges against pixels outside the study-area mask are
    excluded.
    """

    core_mask = (
        (raster_data == CORE_HABITAT_CLASS)
        & valid_mask
    )

    connectivity_structure = np.ones(
        (3, 3),
        dtype=np.uint8,
    )

    labelled_patches, number_of_patches = ndimage.label(
        core_mask,
        structure=connectivity_structure,
    )

    patch_pixel_counts = np.bincount(
        labelled_patches.ravel()
    )[1:]

    patch_pixel_counts = patch_pixel_counts[
        patch_pixel_counts > 0
    ]

    patch_areas_km2 = (
        patch_pixel_counts
        * PIXEL_AREA_KM2
    )

    total_habitat_area_km2 = (
        core_mask.sum()
        * PIXEL_AREA_KM2
    )

    if number_of_patches > 0:

        mean_patch_area_km2 = patch_areas_km2.mean()
        median_patch_area_km2 = np.median(
            patch_areas_km2
        )
        largest_patch_area_km2 = patch_areas_km2.max()

    else:

        mean_patch_area_km2 = 0
        median_patch_area_km2 = 0
        largest_patch_area_km2 = 0

    patch_density_per_100_km2 = (
        number_of_patches
        / ANALYSIS_AREA_KM2
        * 100
    )

    largest_patch_index = (
        largest_patch_area_km2
        / ANALYSIS_AREA_KM2
        * 100
    )

    horizontal_edges = np.sum(
        valid_mask[:, :-1]
        & valid_mask[:, 1:]
        & (
            core_mask[:, :-1]
            != core_mask[:, 1:]
        )
    )

    vertical_edges = np.sum(
        valid_mask[:-1, :]
        & valid_mask[1:, :]
        & (
            core_mask[:-1, :]
            != core_mask[1:, :]
        )
    )

    total_edge_length_metres = (
        horizontal_edges + vertical_edges
    ) * PIXEL_SIZE_METRES

    total_edge_length_km = (
        total_edge_length_metres / 1000
    )

    analysis_area_hectares = (
        ANALYSIS_AREA_KM2 * 100
    )

    edge_density_m_per_ha = (
        total_edge_length_metres
        / analysis_area_hectares
    )

    metrics = {
        "year": year,
        "core_habitat_area_km2": round(
            total_habitat_area_km2,
            3,
        ),
        "number_of_patches": int(
            number_of_patches
        ),
        "patch_density_per_100_km2": round(
            patch_density_per_100_km2,
            3,
        ),
        "mean_patch_area_km2": round(
            mean_patch_area_km2,
            4,
        ),
        "median_patch_area_km2": round(
            median_patch_area_km2,
            4,
        ),
        "largest_patch_area_km2": round(
            largest_patch_area_km2,
            3,
        ),
        "largest_patch_index_percent": round(
            largest_patch_index,
            3,
        ),
        "total_edge_length_km": round(
            total_edge_length_km,
            3,
        ),
        "edge_density_m_per_ha": round(
            edge_density_m_per_ha,
            3,
        ),
    }

    patch_table = pd.DataFrame(
        {
            "year": year,
            "patch_id": np.arange(
                1,
                number_of_patches + 1,
            ),
            "patch_pixels": patch_pixel_counts,
            "patch_area_km2": patch_areas_km2,
        }
    )

    return metrics, patch_table

In [5]:
landscape_metric_records = []
patch_tables = []

for year, raster_data in landuse_arrays.items():

    metrics, patch_table = (
        calculate_core_habitat_metrics(
            raster_data=raster_data,
            valid_mask=reference_mask,
            year=year,
        )
    )

    landscape_metric_records.append(metrics)
    patch_tables.append(patch_table)


core_habitat_landscape_metrics = pd.DataFrame(
    landscape_metric_records
)

core_habitat_patch_areas = pd.concat(
    patch_tables,
    ignore_index=True,
)


METRICS_OUTPUT_PATH = (
    TABLES_DIR
    / "core_habitat_landscape_metrics.csv"
)

PATCH_OUTPUT_PATH = (
    TABLES_DIR
    / "core_habitat_patch_areas.csv"
)


core_habitat_landscape_metrics.to_csv(
    METRICS_OUTPUT_PATH,
    index=False,
)

core_habitat_patch_areas.to_csv(
    PATCH_OUTPUT_PATH,
    index=False,
)


display(core_habitat_landscape_metrics)

print("\nLandscape metrics saved:")
print(METRICS_OUTPUT_PATH)

print("\nPatch-area table saved:")
print(PATCH_OUTPUT_PATH)

,year,core_habitat_area_km2,number_of_patches,patch_density_per_100_km2,mean_patch_area_km2,median_patch_area_km2,largest_patch_area_km2,largest_patch_index_percent,total_edge_length_km,edge_density_m_per_ha
0,1990,21.512,701,91.889,0.0307,0.0019,6.597,0.865,438.400,5.747
1,2012,42.752,1351,177.093,0.0316,0.0025,11.136,1.460,978.675,12.829
2,2024,54.408,2570,336.883,0.0212,0.0012,13.093,1.716,1497.675,19.632



Landscape metrics saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\core_habitat_landscape_metrics.csv

Patch-area table saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\core_habitat_patch_areas.csv


In [6]:
METRIC_COLUMNS = [
    "core_habitat_area_km2",
    "number_of_patches",
    "patch_density_per_100_km2",
    "mean_patch_area_km2",
    "median_patch_area_km2",
    "largest_patch_area_km2",
    "largest_patch_index_percent",
    "total_edge_length_km",
    "edge_density_m_per_ha",
]

METRIC_CHANGE_PERIODS = {
    "1990_2012": (1990, 2012),
    "2012_2024": (2012, 2024),
    "1990_2024": (1990, 2024),
}

metric_change_records = []

metrics_by_year = (
    core_habitat_landscape_metrics
    .set_index("year")
)

for period, (start_year, end_year) in METRIC_CHANGE_PERIODS.items():

    for metric in METRIC_COLUMNS:

        start_value = float(
            metrics_by_year.loc[start_year, metric]
        )

        end_value = float(
            metrics_by_year.loc[end_year, metric]
        )

        absolute_change = end_value - start_value

        if start_value != 0:
            percentage_change = (
                absolute_change / start_value * 100
            )
        else:
            percentage_change = np.nan

        metric_change_records.append(
            {
                "period": period,
                "start_year": start_year,
                "end_year": end_year,
                "metric": metric,
                "start_value": start_value,
                "end_value": end_value,
                "absolute_change": round(
                    absolute_change,
                    4,
                ),
                "percentage_change": round(
                    percentage_change,
                    2,
                ),
            }
        )


landscape_metric_changes = pd.DataFrame(
    metric_change_records
)

METRIC_CHANGE_OUTPUT_PATH = (
    TABLES_DIR
    / "core_habitat_landscape_metric_changes.csv"
)

landscape_metric_changes.to_csv(
    METRIC_CHANGE_OUTPUT_PATH,
    index=False,
)

display(landscape_metric_changes)

print("\nMetric-change table saved:")
print(METRIC_CHANGE_OUTPUT_PATH)

,period,start_year,end_year,metric,start_value,end_value,absolute_change,percentage_change
0,1990_2012,1990,2012,core_habitat_area_km2,21.5120,42.7520,21.2400,98.74
1,1990_2012,1990,2012,number_of_patches,701.0000,1351.0000,650.0000,92.72
2,1990_2012,1990,2012,patch_density_per_100_km2,91.8890,177.0930,85.2040,92.72
3,1990_2012,1990,2012,mean_patch_area_km2,0.0307,0.0316,0.0009,2.93
4,1990_2012,1990,2012,median_patch_area_km2,0.0019,0.0025,0.0006,31.58
5,1990_2012,1990,2012,largest_patch_area_km2,6.5970,11.1360,4.5390,68.80
6,1990_2012,1990,2012,largest_patch_index_percent,0.8650,1.4600,0.5950,68.79
7,1990_2012,1990,2012,total_edge_length_km,438.4000,978.6750,540.2750,123.24
8,1990_2012,1990,2012,edge_density_m_per_ha,5.7470,12.8290,7.0820,123.23
9,2012_2024,2012,2024,core_habitat_area_km2,42.7520,54.4080,11.6560,27.26



Metric-change table saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\core_habitat_landscape_metric_changes.csv


In [7]:
validation_records = []

for year in [1990, 2012, 2024]:

    metric_row = (
        core_habitat_landscape_metrics
        .loc[
            core_habitat_landscape_metrics["year"] == year
        ]
        .iloc[0]
    )

    patch_data = core_habitat_patch_areas[
        core_habitat_patch_areas["year"] == year
    ]

    patch_area_sum = patch_data[
        "patch_area_km2"
    ].sum()

    expected_habitat_area = metric_row[
        "core_habitat_area_km2"
    ]

    expected_patch_count = int(
        metric_row["number_of_patches"]
    )

    validation_records.append(
        {
            "year": year,
            "reported_habitat_area_km2":
                expected_habitat_area,
            "summed_patch_area_km2": round(
                patch_area_sum,
                3,
            ),
            "reported_patch_count":
                expected_patch_count,
            "patch_table_rows": len(patch_data),
            "area_check_passed": np.isclose(
                patch_area_sum,
                expected_habitat_area,
                atol=0.001,
            ),
            "patch_count_check_passed": (
                len(patch_data)
                == expected_patch_count
            ),
        }
    )


landscape_metric_validation = pd.DataFrame(
    validation_records
)

VALIDATION_OUTPUT_PATH = (
    TABLES_DIR
    / "core_habitat_landscape_metric_validation.csv"
)

landscape_metric_validation.to_csv(
    VALIDATION_OUTPUT_PATH,
    index=False,
)

display(landscape_metric_validation)

if not (
    landscape_metric_validation[
        "area_check_passed"
    ].all()
    and landscape_metric_validation[
        "patch_count_check_passed"
    ].all()
):
    raise ValueError(
        "One or more landscape-metric checks failed."
    )

print("\nAll landscape-metric checks passed.")
print("\nValidation table saved:")
print(VALIDATION_OUTPUT_PATH)

,year,reported_habitat_area_km2,summed_patch_area_km2,reported_patch_count,patch_table_rows,area_check_passed,patch_count_check_passed
0,1990,21.512,21.512,701,701,True,True
1,2012,42.752,42.752,1351,1351,True,True
2,2024,54.408,54.408,2570,2570,True,True



All landscape-metric checks passed.

Validation table saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\core_habitat_landscape_metric_validation.csv
